In [0]:
import json
from pathlib import Path
import pandas as pd
import uuid

In [0]:
                                 

# 1. Crear el widget donde esta el path de los datos en crudo
dbutils.widgets.text("raw_root", "/Volumes/workspace/weather_raw/weather", "Raw Root")

raw_root = Path(dbutils.widgets.get("raw_root"))

In [0]:


%sql
-- 2. Borar la dase de datos 
DROP DATABASE IF EXISTS workspace.weather_bronze CASCADE; 

In [0]:

%sql
-- 3. Crear la base de datos
CREATE DATABASE IF NOT EXISTS workspace.weather_bronze
COMMENT 'Capa Bronze: datos crudos procesados for weather'

In [0]:
# 4. Crear la tabla weather_bronze
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.weather_bronze.weather (
  id STRING,
  department STRING,
  year STRING,
  month STRING,
  day STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  generationtime_ms DOUBLE,
  utc_offset_seconds BIGINT,
  timezone STRING,
  timezone_abbreviation STRING,
  elevation DOUBLE,
  daily_units STRING,
  daily STRING
)
""")

In [0]:
# 5. Mostrar la metadata de la tabla creada
spark.table("workspace.weather_bronze.weather").printSchema()

In [0]:

# 6. Cargar los datos del volumen raw

# realizamos la lectura de los datos del volumen
# ordenamos los archivos json

candidate_files = sorted(raw_root.glob("**/weather.json"))
records = []
for json_file in candidate_files:
    payload = json.loads(json_file.read_text(encoding="utf-8"))
    departamento = json_file.parts[-3]
    
    payload["id"] = str(uuid.uuid4())
    payload["department"] = departamento
    payload["year"] = payload["daily"]["time"][0][0:4]
    payload["month"] = payload["daily"]["time"][0][5:7]
    payload["day"] = payload["daily"]["time"][0][8:10]
    
    # Convertir objetos anidados a JSON string
    payload["daily_units"] = json.dumps(payload["daily_units"])
    payload["daily"] = json.dumps(payload["daily"])
    records.append(payload)

# A partir del arreglo creamos el dataframe spark 
df_spark = spark.createDataFrame(records)
df_spark.printSchema()


In [0]:
# 7. Usar overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.weather_bronze.weather")

In [0]:
%sql 
select * from workspace.weather_bronze.weather limit 10 